# STT 04 — Le modèle complet : Transformer + CTC, section par section

Ce notebook contient **tout le code du modèle** (le même que `stt/model.py`), découpé dans l'**ordre du pipeline**. C'est le squelette du modèle : chaque section construit une pièce, explique **à quoi elle sert** et **pourquoi on l'a faite ainsi**, puis la **vérifie** sur de fausses données.

**Ce que fait le modèle :** il reçoit le spectrogramme log-Mel d'un clip audio et renvoie, pour chaque instant, la probabilité de chaque caractère. La loss **CTC** compare ces probabilités à la phrase attendue, sans avoir besoin de savoir à quel instant chaque lettre est prononcée.

```
features log-Mel (B, T, 80)
   │  1. ConvSubsampling              → (B, T/4, 144)  le temps est divisé par ~4
   │  2. SinusoidalPositionalEncoding → (B, T/4, 144)  on ajoute l'ordre des instants
   │  3. 12 × TransformerBlock        → (B, T/4, 144)  chaque instant prend du contexte
   │       ├─ 3a. masque du padding + attention
   │       ├─ 3b. attention multi-tête
   │       ├─ 3c. feed-forward
   │       └─ 3d. bloc complet (pre-norm + connexions résiduelles)
   │  4. LayerNorm finale + output head → (B, T/4, 50)  un score par caractère + blank
   │  5. log_softmax                  → log-probabilités, ce qu'attend la loss CTC
   ▼
loss CTC ← phrase attendue (indices des caractères)
```

**Notations :** `B` = taille du batch (nombre de clips), `T` = nombre d'instants (un toutes les 10 ms), `T'` ≈ T/4 après le sous-échantillonnage, `d_model` = taille des vecteurs dans le Transformer (144).

| Section | Contenu |
|---|---|
| 0 | Imports et vocabulaire |
| 1 | Sous-échantillonnage par convolutions |
| 2 | Encodage des positions |
| 3a | Masque du padding et attention (écrite à la main) |
| 3b | Attention multi-tête |
| 3c | Feed-forward |
| 3d | Bloc Transformer complet |
| 4–5 | Le modèle complet : sortie et log-probabilités |
| 6 | Le pipeline d'apprentissage : forward → loss → backpropagation → optimizer |
| 7 | Récapitulatif : hyperparamètres et liens avec le journal |

Les explications détaillées de chaque notion sont dans `NOTIONS_THEORIQUES.md`, et les décisions (avec leurs raisons) dans `JOURNAL_PROJET.md`.

## 0. Imports et vocabulaire

**À quoi ça sert :** charger PyTorch et le **vocabulaire** du modèle.

- `torch` : les tenseurs (tableaux de nombres) et le calcul des gradients.
- `torch.nn` (`nn`) : les briques des réseaux de neurones (couches `Linear`, `Conv2d`, `LayerNorm`…).
- `torch.nn.functional` (`F`) : les mêmes opérations, sous forme de fonctions.
- `VOCAB` vient de `stt/text.py` : le **blank** du CTC (indice 0), suivi de nos **49 caractères** (26 lettres, 16 lettres accentuées ou ligatures, l'espace et 6 signes). Sa taille, **50**, fixe le nombre de scores produits à chaque instant.

**Pourquoi un blank ?** Le CTC en a besoin pour dire « rien de nouveau à cet instant », et pour séparer deux lettres identiques (le « ll » de « belle »). Voir la notion 9 (le CTC).

In [1]:
import math

import torch
import torch.nn.functional as F
from torch import nn

from stt.text import VOCAB      # blank + 49 caractères (décision : EDA point 2)

torch.manual_seed(0)            # mêmes nombres aléatoires à chaque exécution : résultats reproductibles
print(f"Taille du vocabulaire : {len(VOCAB)} (blank = {VOCAB[0]!r} en position 0)")
print("Caractères :", "".join(VOCAB[1:]))

Taille du vocabulaire : 50 (blank = '_' en position 0)
Caractères :  !',-.?abcdefghijklmnopqrstuvwxyzàâæçèéêëîïôùûüÿœ


## 1. Sous-échantillonnage : `(B, T, 80)` → `(B, T/4, d_model)`

*Notions 11 (convolution) et 12 (ReLU).*

**À quoi ça sert :** réduire le nombre d'instants par environ **4**, et transformer les 80 bandes Mel de chaque instant en **un vecteur de taille `d_model`** (144), la « langue » interne du Transformer.

**Comment :** deux **convolutions 2D** (filtres 3×3, pas de 2) glissent sur le spectrogramme comme sur une image. Chaque convolution divise par ~2 le temps **et** les fréquences :
- temps : 466 → 232 → 115 instants ;
- fréquences : 80 → 39 → 19 bandes.

Une couche `Linear` projette ensuite les `144 canaux × 19 bandes` de chaque instant vers 144 valeurs.

**Pourquoi :**
1. **Le coût de l'attention croît avec le carré du nombre d'instants.** 466² ≈ 217 000 scores contre 115² ≈ 13 000 : **16 fois moins** de calcul et de mémoire.
2. **Les convolutions détectent des motifs locaux** du spectrogramme (le souffle d'un « s », une transition consonne → voyelle) avant que l'attention ne s'occupe du contexte.
3. **Pourquoi ÷ 4 et pas ÷ 8 ?** C'est la **contrainte du CTC** mesurée à l'EDA (point 5). Il faut au moins un instant par caractère : nos phrases en demandent jusqu'à 22,6 par seconde. Après ÷ 4, il en reste 25 par seconde, donc ça passe. Après ÷ 8, il en resterait 12,5, et 44 % des clips deviendraient impossibles.

**Les longueurs aussi :** chaque clip du batch a sa vraie longueur (avant padding). On lui applique la même réduction, sinon le CTC et le masque d'attention utiliseraient des longueurs fausses.

**Formule de la longueur de sortie** d'une convolution (filtre 3, pas 2, sans padding) : ⌊(entrée − 3) / 2⌋ + 1.

In [2]:
def conv_out_length(length):
    # Longueur de sortie d'une convolution de filtre 3, pas 2, sans padding : ⌊(entrée − 3) / 2⌋ + 1
    # Exemple : 466 instants -> 232. Fonctionne aussi sur un tenseur de longueurs.
    return (length - 3) // 2 + 1


class ConvSubsampling(nn.Module):
    # Divise le nombre d'instants par ~4 avec deux convolutions 2D de pas 2

    def __init__(self, n_mels, channels, d_model):
        # n_mels   : nombre de bandes Mel en entrée (80)
        # channels : nombre de filtres par convolution, donc de canaux en sortie
        # d_model  : taille des vecteurs à l'intérieur du Transformer
        super().__init__()  # obligatoire : initialise la partie nn.Module

        self.conv = nn.Sequential(
            # 1re convolution : le spectrogramme (1 canal) passe dans `channels` filtres 3×3.
            # Le pas de 2 divise le temps ET les fréquences par ~2 (466×80 -> 232×39).
            nn.Conv2d(1, channels, kernel_size=3, stride=2),
            # ReLU : remplace les valeurs négatives par 0 (non-linéarité).
            # Sans elle, deux convolutions de suite équivaudraient à UNE seule opération linéaire.
            nn.ReLU(),
            # 2e convolution : combine les motifs détectés par la 1re (232×39 -> 115×19).
            nn.Conv2d(channels, channels, kernel_size=3, stride=2),
            nn.ReLU(),
        )

        # Nombre de bandes de fréquence restant après les deux convolutions : 80 -> 39 -> 19
        freq_out = conv_out_length(conv_out_length(n_mels))

        # Chaque instant est décrit par channels × freq_out valeurs ;
        # on les projette vers la taille interne du modèle (d_model).
        self.linear = nn.Linear(channels * freq_out, d_model)

    def forward(self, x, lengths):
        # x       : (B, T, 80) features log-Mel paddées
        # lengths : (B,) vraies longueurs avant padding
        x = x.unsqueeze(1)                       # (B, 1, T, 80) : Conv2d attend (batch, canaux, hauteur, largeur)
        x = self.conv(x)                         # (B, C, T', 19), avec T' ≈ T / 4
        b, c, t, f = x.shape                     # on récupère les 4 tailles
        x = x.permute(0, 2, 1, 3)                # (B, T', C, 19) : le temps en 2e position
        x = x.reshape(b, t, c * f)               # (B, T', C×19) : chaque instant = un seul vecteur
        x = self.linear(x)                       # (B, T', d_model)
        lengths = conv_out_length(conv_out_length(lengths))   # même réduction pour les vraies longueurs
        return x, lengths

**Vérification :** un faux batch de 2 clips (466 et 300 instants, le second complété par du padding jusqu'à 466). On s'attend à `(2, 115, 144)` en sortie et à des longueurs `[115, 74]` : 300 → (300 − 3) // 2 + 1 = 149 → (149 − 3) // 2 + 1 = 74.

In [3]:
features = torch.randn(2, 466, 80)              # 2 faux clips de 4,66 s (466 instants × 80 bandes)
feature_lengths = torch.tensor([466, 300])      # le 2e ne dure que 3 s : le reste est du padding

subsampling = ConvSubsampling(n_mels=80, channels=144, d_model=144)
x, lengths = subsampling(features, feature_lengths)
print("Entrée :", tuple(features.shape), "-> sortie :", tuple(x.shape))
print("Longueurs :", feature_lengths.tolist(), "->", lengths.tolist())

Entrée : (2, 466, 80) -> sortie : (2, 115, 144)
Longueurs : [466, 300] -> [115, 74]


## 2. Encodage des positions : `(B, T', d_model)` → `(B, T', d_model)`

*Notion 13.*

**À quoi ça sert :** indiquer au modèle **où se trouve chaque instant** dans le clip.

**Pourquoi :** l'attention compare tous les instants entre eux, mais elle ne connaît **pas leur ordre**. Sans encodage des positions, « la » et « al » se ressembleraient pour elle, et le modèle ne saurait pas quel son vient avant quel autre.

**Comment (Transformer original, 2017) :** on **ajoute** à chaque instant un vecteur fixe, fait de sinus et de cosinus de fréquences différentes :
- les premières dimensions varient **vite**, et distinguent des instants voisins ;
- les dernières varient **lentement**, et situent l'instant dans le clip entier.

PE(pos, 2i) = sin(pos / 10000^(2i/d_model)) et PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model)).

**Détail :** ce tableau est un **buffer** (`register_buffer`). Il suit le modèle sur le GPU, mais il n'est **pas appris** : ce ne sont pas des paramètres.

**Taille maximale :** 10 000 positions, soit 400 s après sous-échantillonnage (25 instants par seconde). C'est très au-dessus de nos clips.

In [4]:
class SinusoidalPositionalEncoding(nn.Module):
    # Ajoute à chaque instant un vecteur qui encode sa position (Transformer original, 2017)

    def __init__(self, d_model, max_len=10000):
        # max_len : nombre maximum d'instants (après sous-échantillonnage, 25 par seconde)
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)          # 0, 1, 2, ... en colonne : (max_len, 1)

        # Une fréquence par paire de dimensions : rapide pour les premières, lente pour les dernières.
        # C'est 1 / 10000^(2i / d_model), calculé avec exp et log pour éviter de trop grands nombres.
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))

        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)   # dimensions paires : sinus
        pe[:, 1::2] = torch.cos(position * div_term)   # dimensions impaires : cosinus

        # « buffer » : fait partie du module (suit .to("cuda")), mais n'est PAS appris
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x : (B, T', d_model). On ajoute le motif des T' premières positions ;
        # (T', d_model) + (B, T', d_model) par broadcasting : même motif pour tout le batch.
        return x + self.pe[: x.shape[1]]

**Vérification :** la forme ne change pas, et l'encodage des positions n'ajoute **aucun paramètre** appris.

In [5]:
positional_encoding = SinusoidalPositionalEncoding(d_model=144)
y = positional_encoding(x)
print("Forme :", tuple(x.shape), "->", tuple(y.shape))
print("Paramètres appris :", sum(p.numel() for p in positional_encoding.parameters()))
print("Position 0, 4 premières valeurs :", positional_encoding.pe[0, :4].tolist())   # sin(0)=0, cos(0)=1

Forme : (2, 115, 144) -> (2, 115, 144)
Paramètres appris : 0
Position 0, 4 premières valeurs : [0.0, 1.0, 0.0, 1.0]


## 3a. Masque du padding et attention (écrite à la main)

*Notions 10 (padding et masque) et 14 (attention).*

**À quoi ça sert :** l'**attention** permet à chaque instant d'aller chercher de l'information dans **tous les autres instants** du clip. C'est le cœur du Transformer : pour reconnaître un son ambigu, le modèle regarde ce qui l'entoure.

**Comment :** chaque instant produit trois vecteurs :
- une **requête Q**, ce que l'instant cherche ;
- une **clé K**, ce que l'instant contient ;
- une **valeur V**, l'information qu'il transmet.

Attention(Q, K, V) = softmax(Q·Kᵀ / √d_k) · V

1. **Q·Kᵀ** : un score pour chaque paire (instant i, instant j), qui dit à quel point i doit regarder j.
2. **÷ √d_k** : évite des scores trop grands, qui rendraient le softmax trop « pointu » et les gradients trop faibles.
3. **Masque :** les instants de **padding** (les 0 ajoutés pour égaliser les longueurs dans le batch) reçoivent un score de −∞. Après le softmax, leur poids vaut exactement 0 : aucun instant ne « regarde » du vide.
4. **softmax** : pour chaque instant i, les poids deviennent positifs et leur somme vaut 1.
5. **· V** : chaque instant reçoit une moyenne pondérée des valeurs, c'est-à-dire un résumé du contexte.

**Pourquoi l'écrire à la main ?** Pour comprendre le calcul. On vérifie ensuite qu'elle donne **le même résultat** que la version optimisée de PyTorch, celle utilisée pour l'entraînement.

In [6]:
def make_padding_mask(lengths, max_len):
    # Masque booléen (B, T) : True pour un vrai instant, False pour du padding.
    # Exemple : lengths = [3, 1], max_len = 3  ->  [[True, True, True], [True, False, False]]
    positions = torch.arange(max_len, device=lengths.device)   # (T,) : 0, 1, 2, ...
    return positions[None, :] < lengths[:, None]               # (1, T) < (B, 1) -> (B, T) par broadcasting


def scaled_dot_product_attention(q, k, v, mask):
    # Attention : softmax(Q·Kᵀ / √d_k) · V, en ignorant les instants de padding.
    # q, k, v : (B, T, d_k)   mask : (B, T), True = vrai instant
    d_k = q.shape[-1]

    # Score de chaque paire (instant i, instant j) : (B, T, d_k) @ (B, d_k, T) -> (B, T, T)
    scores = q @ k.transpose(-2, -1) / math.sqrt(d_k)

    # Aucun instant ne doit regarder le padding : mask[:, None, :] a la forme (B, 1, T),
    # on masque des COLONNES (les instants regardés). ~ inverse le masque : padding -> -∞
    scores = scores.masked_fill(~mask[:, None, :], float("-inf"))

    # Pour chaque instant i, des poids positifs qui somment à 1 (le padding reçoit 0)
    weights = torch.softmax(scores, dim=-1)

    # Moyenne pondérée des valeurs : chaque instant reçoit un résumé du contexte
    return weights @ v, weights

**Vérification :**
- le masque est correct ;
- le padding reçoit un poids nul ;
- les poids somment à 1 ;
- notre version donne le même résultat que celle de PyTorch.

In [7]:
print(make_padding_mask(torch.tensor([3, 1]), 3))

q, k, v = torch.randn(3, 2, 5, 8).unbind(0)            # 2 clips, 5 instants, d_k = 8
mask = make_padding_mask(torch.tensor([5, 3]), 5)      # le 2e clip n'a que 3 vrais instants
out, weights = scaled_dot_product_attention(q, k, v, mask)
print("Poids donnés au padding (clip 2) :", weights[1, :, 3:].abs().max().item())
print("Somme des poids par instant :", weights.sum(-1)[0].tolist())

reference = F.scaled_dot_product_attention(q, k, v, attn_mask=mask[:, None, :])
print("Écart maximum avec PyTorch :", (out - reference).abs().max().item())

tensor([[ True,  True,  True],
        [ True, False, False]])
Poids donnés au padding (clip 2) : 0.0
Somme des poids par instant : [1.0, 1.0, 0.9999998807907104, 1.0, 1.0]
Écart maximum avec PyTorch : 1.1920928955078125e-07


## 3b. Attention multi-tête

*Notion 15.*

**À quoi ça sert :** au lieu d'une seule attention, on en fait **plusieurs en parallèle** : ce sont les **têtes**. Chacune travaille sur un morceau du vecteur. Avec 4 têtes, chacune a 144 / 4 = **36 dimensions**.

**Pourquoi :** chaque tête peut se **spécialiser**. Une tête peut regarder les instants voisins (la transition entre deux sons), une autre un contexte plus lointain (la syllabe précédente). Une seule attention devrait faire un compromis entre tout ça.

**Comment :**
1. Une seule couche `Linear` produit Q, K et V d'un coup (3 × 144 sorties), ce qui est plus rapide.
2. On **découpe** en têtes : `(B, T, 144)` → `(B, 4, T, 36)`. Chaque tête est traitée comme une séquence indépendante.
3. On calcule l'attention dans chaque tête.
4. On **recolle** les têtes `(B, T, 144)`, puis une couche `Linear` les recombine.

**`use_pytorch_kernel` :** `True` utilise la version optimisée de PyTorch, plus rapide (c'est celle de l'entraînement). `False` utilise notre version de la section 3a, pour vérifier.

In [8]:
class MultiHeadSelfAttention(nn.Module):
    # Plusieurs attentions en parallèle (têtes), chacune sur un morceau du vecteur

    def __init__(self, d_model, n_heads, use_pytorch_kernel=True):
        super().__init__()
        assert d_model % n_heads == 0, "d_model doit être divisible par n_heads"
        self.n_heads = n_heads
        self.d_k = d_model // n_heads                 # dimensions par tête (144 / 4 = 36)
        self.qkv = nn.Linear(d_model, 3 * d_model)    # produit Q, K et V d'un coup
        self.out = nn.Linear(d_model, d_model)        # recombine les sorties des têtes
        self.use_pytorch_kernel = use_pytorch_kernel  # True : version PyTorch ; False : notre version

    def forward(self, x, mask):
        # x : (B, T, d_model)   mask : (B, T), True = vrai instant
        B, T, d_model = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)        # trois tenseurs (B, T, d_model)

        # Découpage en têtes : (B, T, d_model) -> (B, T, h, d_k) -> (B, h, T, d_k)
        q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)

        head_mask = mask[:, None, :]                  # (B, 1, T) : même règle pour toutes les têtes
        if self.use_pytorch_kernel:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=head_mask[:, None, :, :])
        else:
            out, _ = scaled_dot_product_attention(q, k, v, head_mask)

        # Recollage des têtes : (B, h, T, d_k) -> (B, T, h, d_k) -> (B, T, d_model)
        out = out.transpose(1, 2).reshape(B, T, d_model)
        return self.out(out)

**Vérification :** la forme est conservée, et nos deux versions (la nôtre et celle de PyTorch) donnent la même sortie avec les mêmes poids.

In [9]:
attention = MultiHeadSelfAttention(d_model=144, n_heads=4)
mask = make_padding_mask(lengths, y.shape[1])           # masque des 2 clips de la section 1
out_pytorch = attention(y, mask)
attention.use_pytorch_kernel = False                    # même couche, même poids, notre version
out_ours = attention(y, mask)
print("Forme :", tuple(y.shape), "->", tuple(out_pytorch.shape))
print("Écart maximum entre les deux versions :", (out_pytorch - out_ours).abs().max().item())

Forme : (2, 115, 144) -> (2, 115, 144)
Écart maximum entre les deux versions : 2.980232238769531e-07


## 3c. Feed-forward

*Notion 16.*

**À quoi ça sert :** **transformer** l'information de chaque instant, **séparément** des autres.

**Pourquoi :** l'attention **mélange** l'information entre les instants, et le feed-forward la **traite** pour chaque instant. Les deux rôles se complètent.

**Comment :** `Linear(144 → 576)` → `GELU` → `Dropout` → `Linear(576 → 144)`.
- **Élargir à 576** (4 × `d_model`, le ratio standard du Transformer) donne plus de capacité de calcul.
- **GELU** est une non-linéarité. Sans elle, deux `Linear` de suite reviendraient à une seule.
- **Dropout (10 %)** met au hasard des valeurs à 0 pendant l'entraînement, pour limiter la mémorisation (notion 16). En évaluation, il est désactivé.
- **Revenir à 144** est indispensable pour la connexion résiduelle de la section suivante : on additionne des vecteurs de même taille.

In [10]:
class FeedForward(nn.Module):
    # Réseau appliqué à chaque instant séparément : Linear -> GELU -> Dropout -> Linear

    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),    # élargit : d_model -> d_ff (en général 4 × d_model)
            nn.GELU(),                   # non-linéarité (sinon deux Linear = un seul)
            nn.Dropout(dropout),         # met au hasard 10 % des valeurs à 0 (entraînement seulement)
            nn.Linear(d_ff, d_model),    # revient à d_model (indispensable pour le résiduel)
        )

    def forward(self, x):
        return self.net(x)

In [11]:
feed_forward = FeedForward(d_model=144, d_ff=576, dropout=0.1)
print("Forme :", tuple(y.shape), "->", tuple(feed_forward(y).shape))
print("Paramètres :", f"{sum(p.numel() for p in feed_forward.parameters()):,}")

Forme : (2, 115, 144) -> (2, 115, 144)
Paramètres : 166,608


## 3d. Le bloc Transformer complet (pre-norm + connexions résiduelles)

*Notion 17.*

**À quoi ça sert :** assembler l'attention et le feed-forward en **un bloc**, qu'on empile **12 fois**.

x = x + Dropout(Attention(LayerNorm(x)))
x = x + Dropout(FeedForward(LayerNorm(x)))

**Les trois ingrédients :**
- **LayerNorm** : remet les valeurs de chaque vecteur à une échelle comparable (moyenne 0, écart-type 1), ce qui stabilise l'entraînement.
- **Connexion résiduelle (`x + …`)** : le bloc **ajoute** une correction à `x` au lieu de le remplacer. L'information d'origine est conservée, et les gradients remontent directement vers les premières couches. Sans elle, empiler 12 blocs serait très difficile à entraîner.
- **Pre-norm** : la LayerNorm est appliquée **avant** l'attention et le feed-forward. Le Transformer de 2017 l'appliquait après (post-norm). Le pre-norm est **beaucoup plus stable**, ce qui est précieux pour un modèle entraîné de zéro avec un budget GPU limité.

**Pourquoi 12 blocs ?** C'est la taille « S », choisie pour nos 19 h de données : assez de capacité pour apprendre, sans trop de risque de mémoriser les 275 locuteurs. Les valeurs sont proches de Conformer-S (voir la synthèse des hyperparamètres dans le journal).

In [12]:
class TransformerBlock(nn.Module):
    # Un bloc Transformer pre-norm : x + Attention(LayerNorm(x)), puis x + FFN(LayerNorm(x))

    def __init__(self, d_model, n_heads, d_ff, dropout, use_pytorch_kernel=True):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attention = MultiHeadSelfAttention(d_model, n_heads, use_pytorch_kernel)
        self.norm2 = nn.LayerNorm(d_model)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        # 1. Attention : chaque instant puise de l'information dans les autres (+ résiduel)
        x = x + self.dropout(self.attention(self.norm1(x), mask))
        # 2. Feed-forward : chaque instant est transformé séparément (+ résiduel)
        x = x + self.dropout(self.feed_forward(self.norm2(x)))
        return x

In [13]:
block = TransformerBlock(d_model=144, n_heads=4, d_ff=576, dropout=0.1)
print("Forme :", tuple(y.shape), "->", tuple(block(y, mask).shape))
print("Paramètres d'un bloc :", f"{sum(p.numel() for p in block.parameters()):,}")

Forme : (2, 115, 144) -> (2, 115, 144)
Paramètres d'un bloc : 250,704


## 4–5. Le modèle complet : `SpeechTransformer`

*Notion 18.*

**À quoi ça sert :** **assembler** toutes les pièces, dans l'ordre du pipeline.

1. **Sous-échantillonnage** (section 1) : `(B, T, 80)` → `(B, T', 144)`.
2. **Encodage des positions** (section 2), suivi d'un dropout.
3. **Masque du padding**, calculé une seule fois, puis **12 blocs Transformer** (section 3).
4. **LayerNorm finale + output head** : une couche `Linear(144 → 50)` donne un **score** (*logit*) par symbole du vocabulaire, pour chaque instant. La LayerNorm finale est nécessaire en pre-norm : sinon, le chemin résiduel ne passerait par aucune normalisation. Whisper et les modèles récents font de même.
5. **`log_softmax`** transforme les scores en **log-probabilités**, ce qu'attend `nn.CTCLoss`. Faire `log(softmax)` en une seule opération est plus stable numériquement (cela évite `log(0)`).

**Le modèle renvoie aussi les longueurs réduites :** le CTC doit savoir combien d'instants valides chaque clip possède.

**`nn.ModuleList`** : une liste de modules dont PyTorch **suit les paramètres**. Une liste Python ordinaire les cacherait à l'optimizer, et les 12 blocs ne seraient jamais entraînés.

**Les valeurs par défaut** correspondent à la taille « S » : `d_model` 144, 12 blocs, 4 têtes, `d_ff` 576, dropout 0,1, soit **≈ 3,6 M de paramètres**.

In [14]:
class SpeechTransformer(nn.Module):
    # Features log-Mel (B, T, 80) -> log-probabilités des caractères (B, T/4, 50)

    def __init__(
        self,
        n_mels=80,               # entrée : bandes Mel (stt/audio.py)
        vocab_size=len(VOCAB),   # sortie : 49 caractères + blank = 50 (stt/text.py)
        d_model=144,             # taille des vecteurs dans le Transformer
        n_layers=12,             # nombre de blocs Transformer empilés
        n_heads=4,               # têtes d'attention (144 / 4 = 36 dimensions par tête)
        d_ff=576,                # taille interne du feed-forward (4 × d_model)
        conv_channels=144,       # filtres de chaque convolution du sous-échantillonnage
        dropout=0.1,
        use_pytorch_kernel=True, # attention optimisée de PyTorch (False : notre version)
    ):
        super().__init__()
        self.subsampling = ConvSubsampling(n_mels, conv_channels, d_model)          # étape 1
        self.positional_encoding = SinusoidalPositionalEncoding(d_model)            # étape 2
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(                                                # étape 3
            [TransformerBlock(d_model, n_heads, d_ff, dropout, use_pytorch_kernel) for _ in range(n_layers)]
        )
        self.final_norm = nn.LayerNorm(d_model)                                     # étape 4
        self.output_head = nn.Linear(d_model, vocab_size)

    def forward(self, features, feature_lengths):
        # features : (B, T, 80) log-Mel paddés   feature_lengths : (B,) vraies longueurs
        x, lengths = self.subsampling(features, feature_lengths)       # 1. (B, T', d_model)
        x = self.dropout(self.positional_encoding(x))                  # 2. (B, T', d_model)
        mask = make_padding_mask(lengths, x.shape[1])                  #    (B, T'), une fois pour tous les blocs
        for block in self.blocks:
            x = block(x, mask)                                         # 3. (B, T', d_model)
        x = self.final_norm(x)
        logits = self.output_head(x)                                   # 4. (B, T', 50)
        log_probs = logits.log_softmax(dim=-1)                         # 5. log-probabilités pour le CTC
        return log_probs, lengths

**Vérification :**
- les formes de bout en bout ;
- le **nombre de paramètres** de chaque partie ;
- la conformité avec `stt/model.py`, le code réellement utilisé à l'entraînement : il doit avoir exactement le même nombre de paramètres.

In [15]:
model = SpeechTransformer()
log_probs, out_lengths = model(features, feature_lengths)
print("Entrée :", tuple(features.shape), "-> sortie :", tuple(log_probs.shape))
print("Longueurs de sortie :", out_lengths.tolist())
print("Somme des probabilités à un instant :", log_probs[0, 0].exp().sum().item())   # doit valoir 1

def count(module):
    return sum(p.numel() for p in module.parameters())

print("\nParamètres par partie :")
print(f"  1. sous-échantillonnage : {count(model.subsampling):>10,}")
print(f"  2. positions            : {count(model.positional_encoding):>10,}  (buffer, non appris)")
print(f"  3. 12 blocs             : {count(model.blocks):>10,}  ({count(model.blocks[0]):,} par bloc)")
print(f"  4. norme + output head  : {count(model.final_norm) + count(model.output_head):>10,}")
print(f"  Total                   : {count(model):>10,}")

from stt.model import SpeechTransformer as ModelDuProjet
print("\nMême nombre de paramètres que stt/model.py :", count(model) == count(ModelDuProjet()))

Entrée : (2, 466, 80) -> sortie : (2, 115, 50)
Longueurs de sortie : [115, 74]
Somme des probabilités à un instant : 1.0000001192092896

Paramètres par partie :
  1. sous-échantillonnage :    582,336
  2. positions            :          0  (buffer, non appris)
  3. 12 blocs             :  3,008,448  (250,704 par bloc)
  4. norme + output head  :      7,538
  Total                   :  3,598,322

Même nombre de paramètres que stt/model.py : True


## 6. Le pipeline d'apprentissage : forward → loss → backpropagation → optimizer

*Notions 9 (CTC), 22 (boucle d'entraînement) et 24 (backpropagation, AdamW).*

**À quoi ça sert :** montrer **une itération d'entraînement complète**, exactement celle de la boucle Kaggle (cellule 4), sur un faux batch.

| Étape | Code | Ce qui se passe |
|---|---|---|
| 1. Forward | `model(features, lengths)` | Le modèle prédit des log-probabilités pour chaque instant |
| 2. Loss | `ctc_loss(...)` | Le CTC mesure l'écart avec la phrase attendue, sur tous les alignements possibles |
| 3. Remise à zéro | `optimizer.zero_grad()` | On efface les gradients de l'itération précédente (PyTorch les additionne sinon) |
| 4. Backpropagation | `loss.backward()` | PyTorch calcule, pour **chaque** paramètre, dans quel sens le modifier pour réduire la loss |
| 5. Clipping | `clip_grad_norm_(…, 1.0)` | Si les gradients sont trop grands, on les réduit, pour éviter un pas trop brutal |
| 6. Optimizer | `optimizer.step()` | AdamW modifie les 3,6 M de paramètres d'un petit pas |

**Détails du CTC :**
- `nn.CTCLoss` attend `(T, B, C)` : le temps d'abord. D'où le `transpose(0, 1)`.
- `blank=0` : le blank est en position 0 du vocabulaire.
- `zero_infinity=True` : si un clip est impossible à aligner (trop de caractères pour trop peu d'instants), sa loss serait infinie. On la met à 0 au lieu de faire planter l'entraînement.

**Le résultat attendu :** sur un même batch, la loss doit **baisser** après chaque pas de l'optimizer. C'est le principe du test d'overfitting (notion 22) : si elle ne baisse pas, il y a un bug.

In [16]:
from stt.text import encode

# Fausses cibles : deux courtes phrases encodées en indices (comme le fait stt/dataset.py)
phrases = ["bonjour sarah", "le robot parle"]
targets = torch.nn.utils.rnn.pad_sequence([encode(p) for p in phrases], batch_first=True)
target_lengths = torch.tensor([len(p) for p in phrases])

model = SpeechTransformer()
ctc_loss = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
model.train()                                                     # active le dropout

for step in range(1, 6):
    log_probs, out_lengths = model(features, feature_lengths)                          # 1. forward
    loss = ctc_loss(log_probs.transpose(0, 1), targets, out_lengths, target_lengths)  # 2. loss (T, B, C)
    optimizer.zero_grad()                                                              # 3. efface les anciens gradients
    loss.backward()                                                                    # 4. backpropagation
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)       # 5. clipping
    optimizer.step()                                                                   # 6. mise à jour des poids
    print(f"Pas {step} | loss {loss.item():.3f} | norme du gradient {grad_norm.item():.2f}")

Pas 1 | loss 22.962 | norme du gradient 78.16


Pas 2 | loss 6.999 | norme du gradient 33.16


Pas 3 | loss 5.430 | norme du gradient 5.08


Pas 4 | loss 4.996 | norme du gradient 2.64


Pas 5 | loss 4.953 | norme du gradient 9.24


**Et pour lire la prédiction ?** On passe en mode évaluation (`model.eval()`, sans dropout), puis on applique le **décodage CTC glouton** de `stt/metrics.py`, détaillé dans la notion 20 :
1. à chaque instant, on prend le symbole le plus probable ;
2. on fusionne les répétitions ;
3. on retire les blanks.

Après seulement 5 pas sur de fausses données, la prédiction n'a pas encore de sens : c'est normal.

In [17]:
from stt.metrics import greedy_decode

model.eval()                                              # désactive le dropout
with torch.no_grad():                                     # pas de gradients : plus rapide, moins de mémoire
    log_probs, out_lengths = model(features, feature_lengths)
print(greedy_decode(log_probs, out_lengths))

['r', 'r']


## 7. Récapitulatif

**Le parcours d'un clip dans le modèle**

| Étape | Forme | Rôle |
|---|---|---|
| Entrée | (B, T, 80) | Log-Mel normalisé, un instant toutes les 10 ms |
| 1. Sous-échantillonnage | (B, T/4, 144) | Moins d'instants, motifs locaux, vecteurs de taille `d_model` |
| 2. Positions | (B, T/4, 144) | L'ordre des instants |
| 3. 12 blocs | (B, T/4, 144) | Le contexte (attention) et la transformation (feed-forward) |
| 4. Sortie | (B, T/4, 50) | Un score par symbole |
| 5. log_softmax | (B, T/4, 50) | Les log-probabilités pour le CTC |

**Les hyperparamètres du modèle « S »**

| Hyperparamètre | Valeur | Raison principale |
|---|---|---|
| Sous-échantillonnage | ÷ 4 | Contrainte du CTC (EDA point 5) et coût de l'attention ÷ 16 |
| `d_model` | 144 | Taille S, proportionnée à 19 h de données |
| Blocs | 12 | Taille S, proche de Conformer-S |
| Têtes | 4 | 36 dimensions par tête |
| `d_ff` | 576 | 4 × `d_model`, le ratio standard |
| Dropout | 0,1 | Valeur du Transformer original, contre l'overfitting |
| Vocabulaire | 50 | 49 caractères + blank (EDA point 2) |

**Résultats obtenus avec ce modèle** (voir le journal) :
- référence, 200 époques : WER dev **72,5 %** ;
- avec SpecAugment, 80 époques : WER dev **70,8 %**, et plus d'overfitting.

**Pour aller plus loin :** `NOTIONS_THEORIQUES.md` (convolution, attention, CTC, backpropagation, AdamW, SpecAugment…) et `JOURNAL_PROJET.md` (chaque décision et sa raison).